# CSE457 - XỬ LÝ ÂM THANH VÀ TIẾNG NÓI
## LAB 1: PHÂN TÍCH VÀ XỬ LÝ TÍN HIỆU ÂM THANH SỐ
**Họ và tên:** Lưu Minh Quân    
**MSSV:** 2351260684    
**Lớp:** CSE457
---
### Mục tiêu bài lab:
Thực hành phân tích và xử lý tín hiệu âm thanh số từ miền thời gian đến miền tần số (FFT/STFT), lọc số FIR, lượng tử hóa PCM và nén dữ liệu âm thanh.


## A. Đọc và kiểm tra dữ liệu âm thanh (Audio I/O & Metadata)
Đọc file âm thanh mẫu từ thư mục cục bộ `audio/input/the_quick_brown_fox_male_female.mp3`, kiểm tra các thuộc tính cơ bản ($F_s$, số kênh, độ dài, sample width, file size) và chuẩn hóa biên độ về miền $[-1, 1]$.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import scipy.signal as signal
from scipy.io import wavfile

# Đảm bảo các thư mục tồn tại
os.makedirs("figures", exist_ok=True)
os.makedirs("audio", exist_ok=True)

# Đường dẫn tệp cục bộ
audio_path = "audio/input/the_quick_brown_fox_male_female.mp3"
file_size_kb = os.path.getsize(audio_path) / 1024.0

# Lấy metadata tệp qua soundfile
info = sf.info(audio_path)
Fs = info.samplerate
channels = info.channels
duration = info.duration

# Tải dữ liệu âm thanh dạng Mono và Stereo (chuẩn hóa float32 về [-1, 1])
x, Fs = librosa.load(audio_path, sr=None, mono=True)
x_stereo, _ = librosa.load(audio_path, sr=None, mono=False)

sample_width = 2 # 16-bit PCM tương đương
bit_depth = 16

# Tính RMS và Peak
rms_mono = np.sqrt(np.mean(x**2))
rms_dbfs = 20 * np.log10(rms_mono)
peak_mono = np.max(np.abs(x))

print("=== THÔNG SỐ ÂM THANH ĐẦU VÀO ===")
print(f"1. Đường dẫn file: {audio_path}")
print(f"2. Kích thước file: {file_size_kb:.2f} KB ({file_size_kb/1024:.3f} MB)")
print(f"3. Tần số lấy mẫu (Fs): {Fs} Hz")
print(f"4. Số kênh (Channels): {channels} ({'Stereo' if channels==2 else 'Mono'})")
print(f"5. Thời lượng (Duration): {duration:.3f} s")
print(f"6. Độ rộng mẫu (Sample Width): {sample_width} bytes ({bit_depth}-bit PCM equivalent)")
print(f"7. Kiểu dữ liệu mảng (dtype): {x.dtype}")
print(f"8. Giá trị đỉnh biên độ (Peak): {peak_mono:.4f}")
print(f"9. Mức năng lượng RMS: {rms_mono:.4f} ({rms_dbfs:.2f} dBFS)")


### Nhận xét kỹ thuật (Phần A):
1. **Thông số kỹ thuật:** Tệp âm thanh đầu vào có tần số lấy mẫu $F_s = 22,050\text{ Hz}$, tần số Nyquist tương ứng là $F_{\text{Nyquist}} = 11,025\text{ Hz}$. Tệp có độ rộng mẫu $16\text{-bit}$ với thời lượng khoảng $6.63\text{ s}$.
2. **Biên độ & Mức năng lượng:** Biên độ đỉnh thu được là $Peak \approx 0.7807 < 1.0$, cho thấy tín hiệu hoàn toàn nằm trong dải an toàn và không bị biến dạng do tràn biên độ (clipping). Mức năng lượng RMS đạt $\approx -20.62\text{ dBFS}$, phản ánh mức âm lượng trung bình ổn định của tiếng nói.


## B. Phân tích miền thời gian (Time-Domain Analysis)
Vẽ waveform toàn tệp và đoạn zoom $0.5 - 1.0\text{ s}$. Lựa chọn 2 đoạn tín hiệu có đặc tính khác nhau (ví dụ: đoạn âm hữu thanh tiếng nam vs đoạn khoảng lặng/tiếng nữ) để tính toán $Peak$, $RMS$, $Energy$ ($E = \sum x^2[n]$) và kiểm tra nguy cơ clipping.


In [ ]:
# 1. Vẽ đồ thị Waveform toàn tệp và đoạn Zoom (0.5s - 1.0s)
time_full = np.arange(len(x)) / Fs

fig, axes = plt.subplots(2, 1, figsize=(10, 4.8), sharey=True)

# Waveform toàn tệp
axes[0].plot(time_full, x, color='#3b6998', linewidth=0.5)
axes[0].set_title("Waveform toàn bộ tệp âm thanh tiếng nói", fontsize=11)
axes[0].set_ylabel("Biên độ chuẩn hóa", fontsize=9)
axes[0].grid(True, alpha=0.3)

# Zoom đoạn 0.5s - 1.0s
t_start, t_end = 0.5, 1.0
idx_start, idx_end = int(t_start * Fs), int(t_end * Fs)
axes[1].plot(time_full[idx_start:idx_end], x[idx_start:idx_end], color='#f28e2b', linewidth=0.8)
axes[1].set_title("Zoom đoạn tín hiệu tiếng nói (0.5 s – 1.0 s)", fontsize=11)
axes[1].set_xlabel("Thời gian (s)", fontsize=9)
axes[1].set_ylabel("Biên độ chuẩn hóa", fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("figures/waveform_full_and_zoom.png", dpi=300)
plt.show()

# 2. So sánh 02 đoạn tín hiệu có đặc tính khác nhau
# Đoạn 1: Đoạn âm hữu thanh (Voiced segment, t = 0.5s - 1.5s)
# Đoạn 2: Đoạn khoảng lặng / âm vô thanh (Unvoiced/Silence segment, t = 2.5s - 3.5s)
seg1 = x[int(0.5 * Fs) : int(1.5 * Fs)]
seg2 = x[int(2.5 * Fs) : int(3.5 * Fs)]

def calc_time_metrics(seg):
    peak = np.max(np.abs(seg))
    rms = np.sqrt(np.mean(seg**2))
    rms_db = 20 * np.log10(rms) if rms > 0 else -100
    energy = np.sum(seg**2)
    is_clipped = peak >= 0.99
    return peak, rms, rms_db, energy, is_clipped

p1, r1, r_db1, e1, c1 = calc_time_metrics(seg1)
p2, r2, r_db2, e2, c2 = calc_time_metrics(seg2)

df_time = pd.DataFrame({
    "Đại lượng đo": ["Peak (|x|max)", "RMS", "RMS (dBFS)", "Năng lượng tổng E", "Clipping (> 0.99)?"],
    "Đoạn 1 (Voiced: 0.5s - 1.5s)": [f"{p1:.4f}", f"{r1:.4f}", f"{r_db1:.2f} dBFS", f"{e1:.2f}", "Không" if not c1 else "Có"],
    "Đoạn 2 (Unvoiced: 2.5s - 3.5s)": [f"{p2:.4f}", f"{r2:.4f}", f"{r_db2:.2f} dBFS", f"{e2:.2f}", "Không" if not c2 else "Có"]
})

print("=== BẢNG SO SÁNH CÁC ĐẠI LƯỢNG MIỀN THỜI GIAN ===")
try:
    display(df_time)
except NameError:
    print(df_time)


### Nhận xét kỹ thuật (Phần B):
1. **Đặc tính hình dạng sóng:** Đồ thị Zoom ($0.5 - 1.0\text{ s}$) cho thấy rõ tính chu kỳ của các xung thanh môn (glottal pulses) đặc trưng cho nguyên âm hữu thanh (voiced speech), với chu kỳ cơ bản $T_0 \approx 8.3\text{ ms}$ (tương ứng tần số thanh âm $F_0 \approx 120\text{ Hz}$).
2. **Năng lượng & RMS:** Đoạn âm hữu thanh (Đoạn 1) có mức năng lượng $E$ cao vượt trội so với Đoạn 2. Cả 2 đoạn đều có $Peak < 0.99$, khẳng định tín hiệu ghi âm đạt chất lượng tốt và không gặp hiện tượng clipping.


## C. Phân tích miền tần số bằng FFT (Frequency-Domain Analysis)
Thực hiện cắt đoạn tín hiệu ổn định ($0.5 - 1.5\text{ s}$), nhân cửa sổ Hamming và tính FFT bằng thuật toán `rfft`. Sử dụng thuật toán tự động tìm đỉnh (`find_peaks`) để đánh dấu **ít nhất 3 đỉnh phổ nổi bật**. Thực hiện thử nghiệm so sánh 2 mức NFFT ($NFFT = 2048$ và $NFFT = 65536$) để phân biệt giữa khoảng cách bin tần số $\Delta f$ và độ phân giải vật lý thực (true resolution).


In [ ]:
# 1. Cắt đoạn 0.5s - 1.5s, nhân Hamming và tính FFT (NFFT = 65536)
seg_fft = x[int(0.5 * Fs) : int(1.5 * Fs)]
w_ham = np.hamming(len(seg_fft))
seg_w = seg_fft * w_ham

NFFT1 = 65536
X1 = np.fft.rfft(seg_w, n=NFFT1)
f1 = np.fft.rfftfreq(NFFT1, 1 / Fs)

mag1 = np.abs(X1)
mag_db1 = 20 * np.log10(np.maximum(mag1, 1e-12))
mag_db1_rel = mag_db1 - np.max(mag_db1) # Đỉnh cao nhất = 0 dB

# Tìm các đỉnh nổi bật với distance tối thiểu
min_dist = int(50 / (Fs / NFFT1))
peaks, _ = signal.find_peaks(mag_db1_rel, height=-25, distance=min_dist)

# Chọn 4 đỉnh có biên độ cao nhất
top_peaks = peaks[np.argsort(mag_db1_rel[peaks])[-4:]]
top_peaks = np.sort(top_peaks)

plt.figure(figsize=(10, 3.8))
plt.plot(f1, mag_db1_rel, color='#3b6998', linewidth=0.7, label="Phổ FFT (Hamming window)")

# Annotate nhãn các đỉnh
for p in top_peaks:
    freq_val = f1[p]
    amp_val = mag_db1_rel[p]
    plt.plot(freq_val, amp_val, 'ro', markersize=4)
    plt.annotate(f"{freq_val:.1f} Hz ({amp_val:.1f} dB)",
                 (freq_val, amp_val),
                 xytext=(freq_val + 50, amp_val + 2),
                 fontsize=8,
                 arrowprops=dict(arrowstyle="->", color="red", lw=0.8))

plt.title("Phổ biên độ FFT đoạn tiếng nói hữu thanh (Đã đánh dấu các đỉnh nổi bật)", fontsize=11)
plt.xlabel("Tần số (Hz)", fontsize=10)
plt.ylabel("Biên độ tương đối (dB)", fontsize=10)
plt.xlim(0, 4000)
plt.ylim(-60, 5)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig("figures/fft_spectrum.png", dpi=300)
plt.show()

# 2. So sánh 2 mức NFFT (NFFT = 2048 vs NFFT = 65536)
NFFT2 = 2048
X2 = np.fft.rfft(seg_w, n=NFFT2)
f2 = np.fft.rfftfreq(NFFT2, 1 / Fs)

df_nfft = pd.DataFrame({
    "Cấu hình thử nghiệm": ["NFFT Nhỏ (2048 điểm)", "NFFT Lớn (65536 điểm)"],
    "Độ dài khung L": [f"{len(seg_fft)} mẫu ({len(seg_fft)/Fs*1000:.1f} ms)", f"{len(seg_fft)} mẫu ({len(seg_fft)/Fs*1000:.1f} ms)"],
    "Số điểm NFFT": [NFFT2, NFFT1],
    "Khoảng cách bin Δf (Fs/NFFT)": [f"{Fs/NFFT2:.2f} Hz", f"{Fs/NFFT1:.2f} Hz"],
    "Độ phân giải thực (True Res ≈ Fs/L)": [f"{Fs/len(seg_fft):.2f} Hz", f"{Fs/len(seg_fft):.2f} Hz"],
    "Số bin tần số thực thu được": [len(f2), len(f1)]
})

print("=== BẢNG SO SÁNH GIỮA HAI CẤU HÌNH NFFT ===")
try:
    display(df_nfft)
except NameError:
    print(df_nfft)


### Nhận xét kỹ thuật (Phần C):
1. **Cấu trúc đỉnh phổ (Harmonics):** Phổ biên độ xuất hiện các đỉnh rõ nét tại tần số cơ bản $F_0$ và các họa âm liên tiếp, phản ánh cấu trúc điều hòa đặc trưng của giọng nói.
2. **Phân biệt $\Delta f$ và True Resolution:** Tăng $NFFT$ từ $2048$ lên $65536$ giúp giảm khoảng cách bin tần số $\Delta f$ làm đồ thị phổ mịn hơn do được nội suy qua zero-padding. Tuy nhiên, **độ phân giải vật lý thực sự** vẫn chỉ đạt $\Delta f_{\text{true}} \approx F_s / L$, hoàn toàn do độ dài khung tín hiệu $L$ quyết định.


## D. STFT và Spectrogram (Time-Frequency Analysis)
Tạo Spectrogram với cấu hình chuẩn: độ dài khung $L \approx 25\text{ ms}$, bước nhảy hop size $H \approx 10\text{ ms}$. Thực hiện so sánh ít nhất 3 giá trị độ dài khung ($10\text{ ms}, 25\text{ ms}, 50\text{ ms}$) trên biểu đồ 3-panel để phân tích đánh đổi độ phân giải Thời gian - Tần số (Time-Frequency Resolution Trade-off).


In [ ]:
# Biểu đồ so sánh 3-panel cho 3 frame length (10 ms, 25 ms, 50 ms)
fig, axes = plt.subplots(3, 1, figsize=(10, 7.2), sharex=True, sharey=True)

frame_ms_list = [10, 25, 50]
titles = [
    "1. Frame length = 10 ms (Độ phân giải thời gian cao - dải dọc sắc nét, tần số bị nhòe)",
    "2. Frame length = 25 ms (Cân bằng chuẩn giữa thời gian và tần số - Khuyến nghị cho tiếng nói)",
    "3. Frame length = 50 ms (Độ phân giải tần số cao - đường hài ngang rõ nét, thời gian bị nhòe)"
]

for i, ms in enumerate(frame_ms_list):
    L = round((ms / 1000.0) * Fs)
    H = round(0.010 * Fs) # Hop size cố định 10 ms
    noverlap = max(0, L - H)
    
    f_stft, t_stft, S_stft = signal.spectrogram(
        x, fs=Fs, window='hamming',
        nperseg=L, noverlap=noverlap, nfft=2048, mode='magnitude'
    )
    S_db_stft = 20 * np.log10(np.maximum(S_stft, 1e-12))
    
    mesh = axes[i].pcolormesh(t_stft, f_stft, S_db_stft, shading='gouraud', cmap='viridis', vmin=-90, vmax=np.max(S_db_stft))
    axes[i].set_title(titles[i], fontsize=10)
    axes[i].set_ylabel("Tần số (Hz)", fontsize=9)
    axes[i].set_ylim(0, Fs / 2)

axes[2].set_xlabel("Thời gian (s)", fontsize=9)
fig.subplots_adjust(right=0.90)
cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.70])
fig.colorbar(mesh, cax=cbar_ax, label="Biên độ (dB)")

plt.savefig("figures/spectrogram_multiframe.png", dpi=300)
plt.show()


### Nhận xét kỹ thuật (Phần D):
1. **Cửa sổ ngắn ($10\text{ ms}$):** Cung cấp độ phân giải thời gian rất tốt, giúp nhận diện rõ các thời điểm bắt đầu/kết thúc âm (transients) và các vạch dọc ứng với từng xung đóng thanh môn. Tuy nhiên, các vạch hài ngang bị nhòe do độ phân giải tần số kém.
2. **Cửa sổ dài ($50\text{ ms}$):** Cung cấp độ phân giải tần số xuất sắc, hiển thị rõ từng đường hài (harmonic formants) chạy ngang. Tuy nhiên, biên thời gian bị làm mờ (time smearing), khó phát hiện các biến đổi nhanh.
3. **Cửa sổ chuẩn ($25\text{ ms}$):** Đạt sự cân bằng tối ưu (trade-off) giữa miền thời gian và tần số, phù hợp nhất cho bài toán phân tích đặc trưng tiếng nói (STFT/MFCC).


## E. Thí nghiệm cửa sổ (Windowing Comparison)
So sánh giữa cửa sổ Chữ nhật (Rectangular/Boxcar) và cửa sổ Hamming trên cùng một khung tín hiệu $50\text{ ms}$. Phân tích ảnh hưởng đến bề rộng búp sóng chính (main-lobe width), độ suy hao búp sóng bên (side-lobe suppression) và hiện tượng rò rỉ phổ (spectral leakage).


In [ ]:
# Cắt 1 khung 50 ms tín hiệu voiced
frame_len = round(0.050 * Fs)
start_sample = int(0.6 * Fs)
frame_raw = x[start_sample : start_sample + frame_len]

# Tạo cửa sổ Chữ nhật và Hamming
w_rect = np.ones(frame_len)
w_ham = np.hamming(frame_len)

frame_rect = frame_raw * w_rect
frame_ham = frame_raw * w_ham

NFFT_win = 8192
X_rect = np.fft.rfft(frame_rect, n=NFFT_win)
X_ham  = np.fft.rfft(frame_ham,  n=NFFT_win)
f_win  = np.fft.rfftfreq(NFFT_win, 1 / Fs)

mag_rect_db = 20 * np.log10(np.maximum(np.abs(X_rect), 1e-12)) - 20 * np.log10(np.max(np.abs(X_rect)))
mag_ham_db  = 20 * np.log10(np.maximum(np.abs(X_ham), 1e-12))  - 20 * np.log10(np.max(np.abs(X_ham)))

plt.figure(figsize=(10, 3.8))
plt.plot(f_win, mag_rect_db, color='#e15759', linewidth=0.8, label="Cửa sổ Chữ nhật (Rectangular)")
plt.plot(f_win, mag_ham_db,  color='#3b6998', linewidth=0.8, label="Cửa sổ Hamming")

plt.title("So sánh Phổ Biên độ giữa Cửa sổ Chữ nhật và Hamming (Đánh giá Rò rỉ Phổ)", fontsize=11)
plt.xlabel("Tần số (Hz)", fontsize=10)
plt.ylabel("Biên độ tương đối (dB)", fontsize=10)
plt.xlim(0, 3000)
plt.ylim(-75, 5)
plt.grid(True, alpha=0.25)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("figures/window_compare.png", dpi=300)
plt.show()


### Nhận xét kỹ thuật (Phần E):
1. **Spectral Leakage (Rò rỉ phổ):** Cửa sổ Chữ nhật bị gián đoạn mạnh tại 2 biên của khung, tạo ra các búp sóng bên (side-lobes) cao với mức suy hao chỉ $\approx -13\text{ dB}$. Điều này khiến năng lượng bị rò rỉ ra các dải tần lân cận.
2. **Ưu điểm của Hamming:** Cửa sổ Hamming làm mượt tín hiệu về $0$ tại 2 biên, giảm mức búp sóng bên xuống $\approx -43\text{ dB}$, giúp sàn nhiễu phổ hạ xuống dưới $-60\text{ dB}$. Tuy nhiên, búp sóng chính (main-lobe) của Hamming rộng gấp đôi cửa sổ Chữ nhật, làm giảm nhẹ khả năng tách 2 tần số nằm quá sát nhau.


## F. Lọc số (Digital Filtering)
Thiết kế **01 bộ lọc FIR Low-pass** ($f_c = 2000\text{ Hz}$) và **01 bộ lọc FIR High-pass** ($f_c = 3000\text{ Hz}$) với bậc lọc $M = 200$ ($201\text{ taps}$, cửa sổ Hamming). Vẽ đáp ứng tần số $|H(f)|$, áp dụng lọc tín hiệu, bù trễ nhóm ($group\ delay = 100\text{ mẫu}$), vẽ phổ trước/sau lọc và xuất các file âm thanh `.wav` để nghe thử kiểm chứng.


In [ ]:
# 1. Thiết kế bộ lọc FIR Low-pass (2 kHz) và High-pass (3 kHz)
numtaps = 201
fc_lpf = 2000
fc_hpf = 3000

b_lpf = signal.firwin(numtaps, cutoff=fc_lpf, fs=Fs, pass_zero=True, window='hamming')
b_hpf = signal.firwin(numtaps, cutoff=fc_hpf, fs=Fs, pass_zero=False, window='hamming')

w_lpf, H_lpf = signal.freqz(b_lpf, worN=4096, fs=Fs)
w_hpf, H_hpf = signal.freqz(b_hpf, worN=4096, fs=Fs)

plt.figure(figsize=(10, 3.5))
plt.plot(w_lpf, 20*np.log10(np.maximum(np.abs(H_lpf), 1e-12)), color='#f28e2b', linewidth=1.2, label="FIR Low-pass (fc = 2 kHz)")
plt.plot(w_hpf, 20*np.log10(np.maximum(np.abs(H_hpf), 1e-12)), color='#59a14f', linewidth=1.2, label="FIR High-pass (fc = 3 kHz)")

plt.title("Đáp ứng tần số |H(f)| của bộ lọc FIR Low-pass và High-pass (201 taps)", fontsize=11)
plt.xlabel("Tần số (Hz)", fontsize=10)
plt.ylabel("Biên độ (dB)", fontsize=10)
plt.xlim(0, Fs / 2)
plt.ylim(-100, 5)
plt.grid(True, alpha=0.25)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("figures/filter_response.png", dpi=300)
plt.show()

# 2. Áp dụng lọc tín hiệu và bù trễ nhóm (Group Delay = (201-1)/2 = 100 mẫu)
delay = (numtaps - 1) // 2
y_lpf = signal.lfilter(b_lpf, [1.0], x)
y_hpf = signal.lfilter(b_hpf, [1.0], x)

# 3. Xuất file âm thanh .wav để nghe thử
y_lpf_int16 = np.int16(np.clip(y_lpf, -1.0, 1.0) * 32767)
y_hpf_int16 = np.int16(np.clip(y_hpf, -1.0, 1.0) * 32767)

wavfile.write("audio/filtered_lpf_2kHz.wav", Fs, y_lpf_int16)
wavfile.write("audio/filtered_hpf_3kHz.wav", Fs, y_hpf_int16)

print("Đã xuất thành công các tệp âm thanh đã lọc:")
print(" - audio/filtered_lpf_2kHz.wav")
print(" - audio/filtered_hpf_3kHz.wav")

# 4. Vẽ phổ trước và sau khi lọc
seg_orig = x[int(0.5 * Fs) : int(1.5 * Fs)]
seg_lpf  = y_lpf[int(0.5 * Fs) + delay : int(1.5 * Fs) + delay]
seg_hpf  = y_hpf[int(0.5 * Fs) + delay : int(1.5 * Fs) + delay]

NFFT_flt = 16384
w_flt = np.hamming(len(seg_orig))

X_orig = np.fft.rfft(seg_orig * w_flt, n=NFFT_flt)
X_lpf  = np.fft.rfft(seg_lpf  * w_flt, n=NFFT_flt)
X_hpf  = np.fft.rfft(seg_hpf  * w_flt, n=NFFT_flt)
f_flt  = np.fft.rfftfreq(NFFT_flt, 1 / Fs)

ref_val = np.max(np.abs(X_orig))
mag_orig_db = 20 * np.log10(np.maximum(np.abs(X_orig) / ref_val, 1e-12))
mag_lpf_db  = 20 * np.log10(np.maximum(np.abs(X_lpf)  / ref_val, 1e-12))
mag_hpf_db  = 20 * np.log10(np.maximum(np.abs(X_hpf)  / ref_val, 1e-12))

plt.figure(figsize=(10, 3.8))
plt.plot(f_flt, mag_orig_db, color='#9e9e9e', linewidth=0.6, label="Tín hiệu gốc")
plt.plot(f_flt, mag_lpf_db,  color='#f28e2b', linewidth=0.7, label="Sau Low-pass 2 kHz")
plt.plot(f_flt, mag_hpf_db,  color='#59a14f', linewidth=0.7, label="Sau High-pass 3 kHz")

plt.title("Ảnh hưởng của Bộ lọc FIR lên Phổ Tần số Tín hiệu Âm thanh", fontsize=11)
plt.xlabel("Tần số (Hz)", fontsize=10)
plt.ylabel("Biên độ tương đối (dB)", fontsize=10)
plt.xlim(0, Fs / 2)
plt.ylim(-80, 5)
plt.grid(True, alpha=0.25)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig("figures/filter_spectrum_compare.png", dpi=300)
plt.show()


### Nhận xét kỹ thuật (Phần F):
1. **Đáp ứng bộ lọc Low-pass 2 kHz:** Suy giảm mạnh các thành phần tần số $> 2000\text{ Hz}$ (đạt độ suy hao $> 60\text{ dB}$ trong dải dừng). Khi nghe tệp `filtered_lpf_2kHz.wav`, âm thanh trở nên trầm và "tối" (muffled) do mất đi các phụ âm xát tần số cao.
2. **Đáp ứng bộ lọc High-pass 3 kHz:** Loại bỏ toàn bộ các tần số thấp và tần số cơ bản $F_0 < 3000\text{ Hz}$. Khi nghe tệp `filtered_hpf_3kHz.wav`, âm thanh bị mất giọng nền (thanh âm), chỉ còn lại tiếng xì xào của các phụ âm vô thanh.
3. **Bù trễ nhóm (Group Delay):** Bộ lọc FIR pha tuyến tính bậc $200$ gây ra trễ cố định $\tau = 100\text{ mẫu}$ (tương ứng $\approx 4.54\text{ ms}$). Việc dịch trễ $100\text{ mẫu}$ giúp phổ tín hiệu trước và sau lọc khớp chính xác về mặt pha.


## G. Lượng tử hóa, Resampling và Mã hóa (Quantization, Resampling & Coding)
Thực hiện lượng tử hóa tín hiệu đồng đều với các mức bit $B \in \{4, 6, 8, 12, 16\}$, tính SNR thực nghiệm và xuất file `.wav` lượng tử. Thực hiện Resampling hạ tần số lấy mẫu về $16\text{ kHz}$ và $8\text{ kHz}$ (kèm lọc anti-aliasing). Tính tốc độ bit PCM lý thuyết, dung lượng tệp, tỷ lệ nén (Compression Ratio) và % tiết kiệm (Saving %) so với MP3 gốc.


In [ ]:
# 1. Hàm Lượng tử hóa B-bit đều đối xứng
def quantize(x, B):
    qmax = 2**(B - 1) - 1
    return np.round(np.clip(x, -1.0, 1.0) * qmax) / qmax

def snr_db(x, xq):
    e = xq - x
    power_signal = np.sum(x**2)
    power_noise = np.sum(e**2)
    return 10 * np.log10(power_signal / power_noise)

bit_depths = [4, 6, 8, 12, 16]
snr_values = []

for B in bit_depths:
    xq = quantize(x, B)
    s_val = snr_db(x, xq)
    snr_values.append(s_val)
    
    # Export file wav cho 4-bit và 8-bit
    if B in [4, 8]:
        xq_int16 = np.int16(xq * 32767)
        wavfile.write(f"audio/quantized_{B}bit.wav", Fs, xq_int16)

plt.figure(figsize=(7, 3.5))
plt.plot(bit_depths, snr_values, marker='o', color='#59a14f', linewidth=1.2, markersize=6)
for b, s in zip(bit_depths, snr_values):
    plt.annotate(f"{s:.1f} dB", (b, s), xytext=(0, 6), textcoords='offset points', ha='center', fontsize=9)

plt.title("Tỷ số Tín hiệu trên Nhiễu lượng tử (SNR) theo Số Bit/Mẫu B", fontsize=11)
plt.xlabel("Số bit/mẫu B", fontsize=10)
plt.ylabel("SNR lượng tử đo được (dB)", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("figures/quantization_snr.png", dpi=300)
plt.show()

# 2. Resampling về 16 kHz và 8 kHz bằng resample_poly (Tự động lọc chống aliasing)
Fs_16k, Fs_8k = 16000, 8000
x_16k = signal.resample_poly(x, Fs_16k, Fs)
x_8k  = signal.resample_poly(x, Fs_8k,  Fs)

wavfile.write("audio/resampled_16kHz.wav", Fs_16k, np.int16(np.clip(x_16k, -1, 1) * 32767))
wavfile.write("audio/resampled_8kHz.wav",  Fs_8k,  np.int16(np.clip(x_8k,  -1, 1) * 32767))

print("Đã xuất thành công các tệp resampled:")
print(" - audio/resampled_16kHz.wav")
print(" - audio/resampled_8kHz.wav")

# 3. Tính toán Tốc độ Bit (Bit rate) & Tỷ lệ nén (Compression Ratio)
pcm_rate_mono = (Fs * 16 * 1) / 1000.0 # kbps
mp3_rate = (file_size_kb * 8.0) / duration # kbps

pcm_size_kb = (Fs * 16 * 1 * duration) / (8 * 1024.0)
comp_ratio = pcm_size_kb / file_size_kb
saving_pct = (1.0 - (file_size_kb / pcm_size_kb)) * 100.0

df_coding = pd.DataFrame({
    "Thông số kỹ thuật": ["Tốc độ Bit (Bit Rate)", "Kích thước Tệp (File Size)", "Tỷ lệ Nén (Compression Ratio)", "Tiết kiệm Dung lượng (Saving %)"],
    "PCM 16-bit Mono (Gốc lý thuyết)": [f"{pcm_rate_mono:.1f} kbps", f"{pcm_size_kb:.2f} KB ({pcm_size_kb/1024:.2f} MB)", "1 : 1", "0 %"],
    "MP3 Thực tế (Tệp đầu vào)": [f"{mp3_rate:.1f} kbps", f"{file_size_kb:.2f} KB ({file_size_kb/1024:.2f} MB)", f"{comp_ratio:.2f} : 1", f"{saving_pct:.1f} %"]
})

print("=== BẢNG THÔNG SỐ NÉN ÂM THANH ===")
try:
    display(df_coding)
except NameError:
    print(df_coding)

# Biểu đồ cột so sánh Bit rate
labels = ["PCM 16-bit Mono\n(Gốc lý thuyết)", f"MP3 Gốc thực tế\n({mp3_rate:.0f} kbps)", "MP3 128 kbps\n(Tham chiếu)"]
values = [pcm_rate_mono, mp3_rate, 128]
colors = ['#3b6998', '#5b9bd5', '#9e9e9e']

plt.figure(figsize=(7, 3.5))
bars = plt.bar(labels, values, color=colors, width=0.5)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{yval:.0f} kbps", ha='center', va='bottom', fontsize=9)

plt.title("So sánh Tốc độ Bit giữa PCM không nén và MP3", fontsize=11)
plt.ylabel("Tốc độ Bit (kbps)", fontsize=10)
plt.ylim(0, max(values) * 1.2)
plt.tight_layout()
plt.savefig("figures/bitrate_comparison.png", dpi=300)
plt.show()


### Nhận xét kỹ thuật (Phần G):
1. **SNR Lượng tử:** Đúng như lý thuyết $SNR \approx 6B + 4.77\text{ dB}$, SNR đo được tăng gần như tuyến tính với hệ số $\approx 6\text{ dB/bit}$ (đạt $14.1\text{ dB}$ ở $4\text{-bit}$ và $86.5\text{ dB}$ ở $16\text{-bit}$). Khi nghe `quantized_4bit.wav`, tiếng ồn lượng tử (hissing noise) xuất hiện rất rõ nét.
2. **Resampling:** Hạ $F_s$ về $8\text{ kHz}$ thu hẹp dải tần khả dụng xuống $4\text{ kHz}$ (Nyquist). Âm thanh trong `resampled_8kHz.wav` bị mất dải tần cao nhưng tiếng nói vẫn hoàn toàn nghe rõ (đạt chuẩn thoại điện thoại truyền thống).
3. **Hiệu quả nén:** Định dạng MP3 nén với tỷ lệ $comp\_ratio \approx 6.37 : 1$, giúp tiết kiệm $\approx 84.3\%$ dung lượng lưu trữ so với PCM $16\text{-bit}$ không nén mà vẫn giữ nguyên chất lượng cảm nhận thính giác.


## H. Trả lời 7 câu hỏi báo cáo lý thuyết (Mục 6 trong Lab 1.pdf)

### **Câu 1: Giải thích bằng công thức tại sao $F_s = 44.1\text{ kHz}$ chỉ biểu diễn độc lập đến $22.05\text{ kHz}$.**
**Trả lời:**
Theo **Định lý lấy mẫu Nyquist-Shannon**, để khôi phục hoàn toàn một tín hiệu liên tục có tần số cực đại $F_{\max}$ từ chuỗi lấy mẫu rời rạc mà không bị hiện tượng chồng phổ (aliasing), tần số lấy mẫu $F_s$ phải thỏa mãn:
$$F_s \ge 2 F_{\max} \implies F_{\max} \le \frac{F_s}{2}$$
Với $F_s = 44.1\text{ kHz}$, tần số Nyquist là:
$$F_{\text{Nyquist}} = \frac{44.1\text{ kHz}}{2} = 22.05\text{ kHz}$$
Bất kỳ thành phần tần số nào $f > 22.05\text{ kHz}$ sẽ bị dội ngược (alias) về dải $[0, 22.05\text{ kHz}]$ theo công thức $f_{\text{alias}} = |F_s - f|$, do đó $F_s = 44.1\text{ kHz}$ chỉ biểu diễn độc lập và chính xác các tần số đến $22.05\text{ kHz}$.

---

### **Câu 2: Nếu NFFT tăng từ 2048 lên 8192 nhưng frame vẫn dài 25 ms, điều gì thật sự thay đổi và điều gì không?**
**Trả lời:**
- **Điều THẬT SỰ THAY ĐỔI:** Khoảng cách giữa các bin tần số $\Delta f$ giảm đi 4 lần:
  $$\Delta f = \frac{F_s}{NFFT} \implies \Delta f_{2048} = \frac{44100}{2048} \approx 21.53\text{ Hz} \longrightarrow \Delta f_{8192} = \frac{44100}{8192} \approx 5.38\text{ Hz}$$
  Đồ thị phổ trở nên mịn hơn và có nhiều điểm vẽ hơn trên trục tần số (nội suy phổ qua zero-padding).
- **Điều KHÔNG THAY ĐỔI:** **Độ phân giải vật lý thực sự (True Physical Resolution)** không thay đổi! Độ phân giải thực hoàn toàn do độ dài khung thời gian $L = 25\text{ ms}$ và hàm cửa sổ quyết định:
  $$\Delta f_{\text{true}} \approx \frac{F_s}{L} = \frac{1}{0.025\text{ s}} = 40\text{ Hz}$$
  Tăng NFFT bằng zero-padding chỉ nội suy mượt các điểm giữa chứ không tạo thêm thông tin mới để phân tách 2 tần số nằm sát nhau hơn $40\text{ Hz}$.

---

### **Câu 3: Tại sao Hamming giảm spectral leakage so với rectangular nhưng có thể làm các đỉnh gần nhau khó phân tách hơn?**
**Trả lời:**
- **Hamming giảm rò rỉ phổ:** Cửa sổ Hamming triệt tiêu biên độ về gần 0 ở 2 mép khung, loại bỏ sự gián đoạn biên độ. Nhờ đó, biên độ búp sóng bên (side-lobe) của Hamming giảm xuống $-43\text{ dB}$ (so với $-13\text{ dB}$ của Rectangular), hạn chế năng lượng rò rỉ ra các dải tần xa.
- **Khó phân tách các đỉnh gần nhau:** Búp sóng chính (main-lobe) của cửa sổ Hamming có độ rộng $8\pi / L$, rộng gấp đôi búp sóng chính của cửa sổ Rectangular ($4\pi / L$). Búp sóng chính rộng hơn sẽ làm nhòe (smear) dải tần, khiến hai đỉnh phổ có tần số rất sát nhau dễ bị hòa lẫn thành một đỉnh duy nhất.

---

### **Câu 4: Với FIR 201 taps đối xứng tại 44.1 kHz, độ trễ xấp xỉ bao nhiêu mili giây? Độ trễ đó có quan trọng trong xử lý thời gian thực không?**
**Trả lời:**
- **Độ trễ nhóm (Group Delay):** Với bộ lọc FIR pha tuyến tính đối xứng có số taps $M = 201$, độ trễ nhóm cố định là:
  $$\tau = \frac{M - 1}{2} = \frac{201 - 1}{2} = 100\text{ mẫu}$$
  Đổi sang mili giây tại $F_s = 44.1\text{ kHz}$:
  $$t_{\text{delay}} = \frac{100}{44100\text{ Hz}} \approx 2.267.5\text{ ms}$$
- **Ý nghĩa trong xử lý thời gian thực:** Độ trễ $\approx 2.27\text{ ms}$ là rất nhỏ (nằm dưới ngưỡng cảm nhận độ trễ thính giác $10\text{ ms} - 20\text{ ms}$ của con người). Vì vậy, độ trễ này hoàn toàn chấp nhận được trong hầu hết các ứng dụng xử lý âm thanh thời gian thực (live audio, VoIP, trợ thính).

---

### **Câu 5: Từ công thức $SNR_Q$, giải thích ảnh hưởng của $B$ và $\sigma_x$. Tại sao giảm mức tín hiệu đầu vào có thể làm SNR lượng tử giảm?**
**Trả lời:**
Công thức SNR lượng tử lý thuyết cho tín hiệu liên tục có RMS $\sigma_x$ và giá trị cực đại $X_{\max}$:
$$SNR_Q (dB) = 6.02 B + 4.77 - 20 \log_{10}\left(\frac{X_{\max}}{\sigma_x}\right)$$
- **Ảnh hưởng của $B$:** Mỗi bit bổ sung tăng $SNR_Q$ xấp xỉ $6.02\text{ dB}$ do bước lượng tử $\Delta = 2 X_{\max} / 2^B$ giảm đi một nửa, giảm công suất nhiễu lượng tử $P_e = \Delta^2 / 12$.
- **Ảnh hưởng của $\sigma_x$:** Nhiễu lượng tử có biên độ cố định $\Delta$. Khi giảm mức tín hiệu đầu vào (giảm $\sigma_x$), công suất tín hiệu $P_x = \sigma_x^2$ giảm xuống trong khi công suất nhiễu lượng tử $P_e$ giữ nguyên. Tỷ số $X_{\max} / \sigma_x$ tăng lên, làm đại lượng $-20 \log_{10}(X_{\max}/\sigma_x)$ âm nhiều hơn, dẫn đến **$SNR_Q$ bị sụt giảm nghiêm trọng**.

---

### **Câu 6: Một file WAV 16-bit stereo 44.1 kHz dài 60 s có kích thước PCM lý thuyết bao nhiêu MB? So sánh với MP3 128 kbps.**
**Trả lời:**
- **Tốc độ bit PCM lý thuyết:**
  $$R_{\text{PCM}} = 44100\text{ Hz} \times 16\text{ bit} \times 2\text{ kênh} = 1,411,200\text{ bit/s} = 1411.2\text{ kbps}$$
- **Kích thước PCM lý thuyết trong 60 s:**
  $$\text{Size}_{PCM} = \frac{1,411,200 \times 60}{8 \times 1024 \times 1024} \approx 10.094\text{ MB}$$
- **Kích thước MP3 128 kbps trong 60 s:**
  $$\text{Size}_{MP3} = \frac{128,000 \times 60}{8 \times 1024 \times 1024} \approx 0.915\text{ MB}$$
- **So sánh:** File PCM không nén lớn gấp $\approx 11.03$ lần so với file MP3 128 kbps (nén tiết kiệm $\approx 90.9\%$ dung lượng).

---

### **Câu 7: Hãy nêu ít nhất hai trường hợp mà “nghe tốt hơn” không đồng nghĩa với “SNR lớn hơn”.**
**Trả lời:**
1. **Trường hợp 1: Áp dụng Bộ lọc LPF triệt tiêu nhiễu cao tần (Noise Reduction Filtering).**
   Bộ lọc Low-pass loại bỏ nhiễu trắng/nhiễu rít tần số cao làm tín hiệu nghe êm tai và dễ chịu hơn nhiều. Tuy nhiên, LPF cũng cắt bỏ một phần năng lượng tín hiệu gốc ở dải tần cao. Nếu tính SNR so với tín hiệu gốc ban đầu, công suất sai số (phần phổ bị cắt đi) có thể lớn làm $SNR$ tính toán giảm xuống, dù cảm nhận thính giác "nghe tốt hơn".
2. **Trường hợp 2: Mã hóa âm thanh theo mô hình tâm lý thính giác (Perceptual Audio Coding - MP3/AAC).**
   Các thuật toán nén như MP3 cố tình giấu nhiễu lượng tử bên dưới ngưỡng nghe thấy của tai người (Auditory Masking). Độ chênh lệch giữa tín hiệu gốc và tín hiệu nén có thể tạo ra công suất nhiễu $P_e$ khá lớn (dẫn đến $SNR$ nhỏ), nhưng người nghe không thể phát hiện ra nhiễu này, cho cảm giác âm thanh trung thực và "nghe tốt hơn" rất nhiều so với phương pháp lượng tử hóa đều $4\text{-bit}$ có $SNR$ tương đương.
